# HTTPS Downloads of CoastWatch Data 

### Author: Dale Robinson & Madison Richardson

> History | Updated July 2026

## 1. Introduction
Most CoastWatch data can be accessed via an **HTTPS (Hypertext Transfer Protocol Secure)** data server. 

* HTTPS is a standard web protocol for downloading complete files.  
* It is particularly useful for **bulk downloads** (e.g., 10 years of global VIIRS chlorophyll-a data).
* Unlike subsetting services (like ERDDAP or OPeNDAP), HTTPS is best for full-file downloads.

In this tutorial, we will learn **Python methods to download CoastWatch data over HTTPS**.

## 2. Dataset for This Tutorial
We’ll use:
  
**Sea Level Anomaly and Geostrophic Currents, multi-mission, global, optimal interpolation, gridded**

* Open the [dataset documentation page](https://coastwatch.noaa.gov/cwn/products/sea-level-anomaly-and-geostrophic-currents-multi-mission-global-optimal-interpolation.html). 

<img src="ssh_download_options.png" width="500">

## 3. Exploring the HTTPS Directory

1. From the dataset documentation page, scroll down and click on the **HTTPS** link.  
* Subdirectories are organized by year (2015 → present).
* Other datasets may be organized differently (e.g., monthly or by region).

2. Open the **2024** directory. Inside the year folder, you’ll see a list of files. 

3. File naming convention:
    >rads_global_nrt_sla_20240119_20240120_001.nc

Breakdown:
* rads: Radar Altimetry Database System
* global: Global dataset
* nrt: Near real-time
* sla: Sea level anomaly
* 20240119: Date of the data (YYYYMMDD = Jan 19, 2024)
* 20240120: Date processed (YYYYMMDD = Jan 20, 2024)

4. Find the file for January 19, 2024 (rads_global_nrt_sla_20240119_20240120_001.nc). 
* Right click and copy the link to obtain the following download URL for the file.  
[https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/2024/    rads_global_nrt_sla_20240119_20240120_001.nc](https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/2024/rads_global_nrt_sla_20240119_20240120_001.nc)
* Put the URL into a variable called **download_url**.

In [1]:
download_url = (
    "https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/"
    "rads/sla/2024/rads_global_nrt_sla_20240119_20240120_001.nc"
)

## 4. Programmatic Downloads 
We’ll demonstrate using Python (via the `requests` library) 

There is another approach using system tools (`wget` or `curl`) that we demonstrate in the [Appendix](#8-appendix-a).

### FIX APPENDIX LINK ^

### Python Download Function 

In [2]:
import os
import requests
from urllib.parse import urlparse

def download_file_with_python(url: str, output_dir: str) -> str:
    """
    Download a file from an HTTPS URL and save it in the specified directory.

    Args:
        url (str): The HTTPS URL of the file to download.
        output_dir (str): The local directory where the file will be saved.
                          The original file name from the URL is preserved.

    Returns:
        str: The full path to the downloaded file.
    """
    # Extract filename from URL
    filename = os.path.basename(urlparse(url).path)
    if not filename:
        raise ValueError("URL does not contain a valid filename.")

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Full path to save the file
    output_path = os.path.join(output_dir, filename)

    # Download with streaming
    response = requests.get(url, stream=True)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return output_path


#### Usage example

In [3]:
from pathlib import Path

# The download_url variable is defined above 
out_dir = Path("downloads") / "sea_level_anon" / "2024"

saved_file = download_file_with_python(download_url, out_dir)
print(f"File saved to: {saved_file}")

File saved to: downloads/sea_level_anon/2024/rads_global_nrt_sla_20240119_20240120_001.nc


## 5. Bulk Downloads
Because CoastWatch HTTPS directories follow **consistent URL patterns** and **file naming conventions**, we can easily automate downloads. 

For the **Sea Level Anomaly and Geostrophic Currents, multi-mission, global, optimal interpolation, gridded** dataset: 

URLs have the structure:  
>www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/YYYY, where  
* **YYYY** is the coverage year.  

File names have the structure:   
> rads_global_nrt_sla_20240119_20240120_001.nc, where:  
* **20240119** is the date of data coverage, in YYYYMMDD format (Jan. 19, 2024)

### 5.1 Task Scenario
Your goal is to download all June netCDF files for the years 2022 and 2023 from the CoastWatch HTTPS directory  

**Workflow steps**: 
1. **Scrape file URLs** – Write a helper function that collects all .nc file links within a given year directory.
2. **Aggregate results** – Use the function on each year directory to build a full list of file URLs.
3. **Filter by month** – From the combined list, select only the files that correspond to **June** (YYYY06).
4. **Download files** – Save the June files to a local folder on your computer using the **download_file_with_python()** helper function. 

### 5.2 Scraper function that collects .nc file URLs

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_http_files(url, file_ext=None, timeout=10):
    """
    Scrape file names (not directories) from an HTTP directory listing.

    Args:
        url (str): HTTP URL to scrape (trailing slash optional).
        file_ext (str or list[str], optional): File extension(s) to filter 
                                               (e.g., '.nc' or ['.nc', '.txt']).
                                               Defaults to None (all files).
        timeout (int, optional): Timeout in seconds for HTTP request. Defaults to 10.

    Returns:
        list[str]: List of absolute URLs to files.

    Raises:
        RuntimeError: If the HTTP request fails.

    Example:
        >>> scrape_http_files(
        ...     "https://coastwatch.noaa.gov/pub/socd2/coastwatch/sst/ran/l3s/leo/daily",
        ...     file_ext=".nc"
        ... )
        ['https://coastwatch.noaa.gov/.../sst_20250215.nc']
    """
    # Ensure the URL ends with a slash
    if not url.endswith('/'):
        url += '/'

    # Request the directory listing webpage
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(f"Failed to fetch {url}: {e}")

    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Convert a single extension into a list
    if isinstance(file_ext, str):
        file_ext = [file_ext]

    # Standardize extensions for comparison
    if file_ext:
        file_ext = [ext.lower() for ext in file_ext]

    # Store matching file URLs
    files = []

    # Examine each hyperlink in the directory listing
    for link in soup.find_all('a'):
        href = link.get('href')

        # Skip invalid and navigation links
        if not href or href.startswith(('?', '/', '../')):
            continue

        # Skip subdirectories
        if href.endswith('/'):
            continue

        # Keep files matching the requested extension(s)
        if not file_ext or any(href.lower().endswith(ext) for ext in file_ext):
            files.append(urljoin(url, href))

    # Return sorted file URLs
    return sorted(files)

### 5.3 Complete Workflow: Bulk Download of June netCDF Files (2022–2023)

In [5]:
import sys
from pathlib import Path

# Years to download data for
years = [2022, 2023]

# Months to download (June)
months = [6]

# Loop through each year
for year in years:

    # HTTPS directory for the current year
    url_path = f"https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/{str(year)}/"

    # Get all NetCDF file URLs for the year
    files_paths_for_year = scrape_http_files(url_path, file_ext=".nc", timeout=10)

    # Loop through each selected month
    for month in months:

        # Format month as two digits (e.g., 06)
        month_str = f"{month:02d}"

        # Pattern used to identify files for this year/month
        search_pattern = f"rads_global_nrt_sla_{str(year)}{month_str}"

        # Keep only files matching the year/month
        files_for_month = [ln for ln in files_paths_for_year if search_pattern in ln]

        # Download matching files
        for file_to_get in files_for_month:

            # Output directory for downloaded files
            out_dir = Path("downloads") / "sea_level_anon" / str(year) / month_str

            # Display file being downloaded
            print("downloading", file_to_get)

            # Download the file
            saved_file = download_file_with_python(file_to_get, out_dir)

            # Display saved file path
            print(f"File saved to: {saved_file}")

            # Stop after the first file for testing
            # Remove this break to download all files
            print("Comment out break to download all files")
            break

downloading https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/2022/rads_global_nrt_sla_20220601_20220602_001.nc
File saved to: downloads/sea_level_anon/2022/06/rads_global_nrt_sla_20220601_20220602_001.nc
Comment out break to download all files
downloading https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/2023/rads_global_nrt_sla_20230601_20230602_001.nc
File saved to: downloads/sea_level_anon/2023/06/rads_global_nrt_sla_20230601_20230602_001.nc
Comment out break to download all files


## 6. Things to Keep in Mind
* HTTPS directory structures vary between datasets (by year, month, region, etc.).
* File naming conventions also vary — check documentation before automating.
* Directory listings may include files for multiple product types (SST, Chlorophyll...).
    * You may need to filter for the desired product.
* Consider filtering by extension (.nc) to avoid downloading unwanted file types.

## 7. Assignments
### Task 1
Your task is to download the file for the **16th day of each month** between **2020 and 2022**, using a different dataset from the CoastWatch HTTPS directory. Organize the downloaded files into subdirectories by year inside a top-level folder named **geopolar**. 

We’ll use:
  
#### Dataset: NOAA Geo-Polar Blended Global Sea Surface Temperature Analysis

This dataset combines sea surface temperature (SST) observations from multiple **polar-orbiting** and **geostationary satellites**, producing a global SST product at **0.05° (~5 km) resolution**.

#### Instructions
1. Open the [dataset documentation page](https://coastwatch.noaa.gov/cwn/products/noaa-geo-polar-blended-global-sea-surface-temperature-analysis-level-4.html).
2. Scroll down to the HTTPS section.
3. Click the Day + Night link to access the data directory.
4. Write Python code to:
    * Scrape the available files,
    * Select the files corresponding to the 16th of each month,
    * Download them into a directory structure like:

💡 Hint: You can adapt the scraping and downloading functions developed in the previous sections.

**Good luck, and have fun using your new data access skills!**

### Task 2 - Extra challenging task (optional)
Your task is the same as for Task 1, but this time use the **S3A-OLCI** dataset from the [NASA OceanColor HTTPS directory](https://oceandata.sci.gsfc.nasa.gov/directdataaccess/Level-3%20Mapped/). 

The dataset contains ocean color data from **Ocean and Land Colour Instrument (OLCI)** about the **European Space Agency's (ESA) Sentinel-3A** satellite.

## 8. Appendix A

### Using `wget` or `curl` for programmatic downloads


In [6]:
import os
import subprocess

def download_with_wget_curl(url: str, output_dir, program: str) -> int:
    """
    Download a file from a URL using either the system ``wget`` or ``curl``
    command-line utility.

    Args:
        url (str):
            The HTTP or HTTPS URL of the file to download.

        output_dir (str or pathlib.Path):
            Directory where the downloaded file will be saved. If the
            directory does not already exist, it will be created
            automatically. Accepts either a string path or a ``Path`` object.

        program (str):
            Download utility to use. Must be either ``"wget"`` or ``"curl"``.

    Returns:
        int:
            Return code from the executed download command. A value of ``0``
            indicates a successful download, while non-zero values indicate
            that an error occurred during the download process.
    """
    allowed = {"wget", "curl"}
    if program not in allowed:
        raise ValueError(f"Invalid program '{program}'. Must be one of {allowed}")
    # Ensure the output directory exists
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    download_path = output_dir / Path(url).name

    # Build the wget command
    if program == "wget":
        cmd = ["wget", "-P", str(output_dir), url]
    elif program == "curl":
        cmd = ["curl", "-o", str(download_path), url]

    # Run the command
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"Downloaded {url} → {output_dir}")
    else:
        print(f"Error downloading {url}: {result.stderr.strip()}")

    return result.returncode



### Usage example

In [7]:
# The download_url variable is defined above 

out_dir = Path("downloads") / "sea_level_anon" / "2024"

success = download_with_wget_curl(download_url, out_dir, "curl")

print("zero indicates successfull download", success)

Downloaded https://www.star.nesdis.noaa.gov/data/pub0015/coastwatch/rads/sla/2024/rads_global_nrt_sla_20240119_20240120_001.nc → downloads/sea_level_anon/2024
zero indicates successfull download 0
